In [1]:
# !pip install optuna lightgbm pandas scikit-learn matplotlib

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data (replace with your dataset path)
X_train = pd.read_parquet('temp/X_train.parquet')
X_val = pd.read_parquet('temp/X_val.parquet')


y_train = X_train['TARGET']
y_val = X_val['TARGET']

X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [3]:
y_train.isnull().sum(), y_val.isnull().sum()

(0, 0)

In [4]:
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_val.columns = X_val.columns.str.replace(r'[^\w]', '_', regex=True)

In [5]:
import optuna
import lightgbm as lgb
import numpy as np

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 2e-2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 10, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.8),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        'n_jobs': -1
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=1000,
    )
    
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    return auc


In [6]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Print the best parameters
print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)

[I 2024-12-05 23:24:15,154] A new study created in memory with name: no-name-539ac2bf-35cc-46e6-b721-e823ac23a5c9


[LightGBM] [Warning] min_data_in_leaf is set=193, min_child_samples=150 will be ignored. Current value: min_data_in_leaf=193
[LightGBM] [Warning] min_data_in_leaf is set=193, min_child_samples=150 will be ignored. Current value: min_data_in_leaf=193
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.785923 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 172487
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 1068
[LightGBM] [Warning] min_data_in_leaf is set=193, min_child_samples=150 will be ignored. Current value: min_data_in_leaf=193
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


In [7]:
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

In [8]:
best_params

{'learning_rate': 0.015704338232403883,
 'num_leaves': 85,
 'max_depth': 15,
 'min_data_in_leaf': 156,
 'feature_fraction': 0.4640334733534099,
 'bagging_fraction': 0.7771094415545631,
 'bagging_freq': 6,
 'lambda_l1': 9.68355576275023,
 'lambda_l2': 0.0021699932433295504,
 'min_child_samples': 180,
 'objective': 'binary',
 'metric': 'auc'}

In [9]:
best_params = best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

print("Training the final model with the best parameters")
print(best_params)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=1000
)

Training the final model with the best parameters
{'learning_rate': 0.015704338232403883, 'num_leaves': 85, 'max_depth': 15, 'min_data_in_leaf': 156, 'feature_fraction': 0.4640334733534099, 'bagging_fraction': 0.7771094415545631, 'bagging_freq': 6, 'lambda_l1': 9.68355576275023, 'lambda_l2': 0.0021699932433295504, 'min_child_samples': 180, 'objective': 'binary', 'metric': 'auc'}
[LightGBM] [Warning] min_data_in_leaf is set=156, min_child_samples=180 will be ignored. Current value: min_data_in_leaf=156
[LightGBM] [Warning] min_data_in_leaf is set=156, min_child_samples=180 will be ignored. Current value: min_data_in_leaf=156
[LightGBM] [Info] Number of positive: 226133, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.875354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 172487
[LightGBM] [Info] Number of data points in the train set: 452266, number of used features: 1068
[

In [10]:
y_pred = final_model.predict(X_val)

In [11]:
roc_auc_score(y_val,y_pred)

0.7923351954725212

In [12]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
importance_split = final_model.feature_importance(importance_type='split')
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance,
    'Num_Split': importance_split
}).sort_values(by='Importance', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())


The most important feature is: WEAK_FEATURE with an importance score of 1051030.1580238342

Top 5 Features:
               Feature    Importance  Num_Split
1084      WEAK_FEATURE  1.051030e+06        848
939   ext_sources_mean  3.682881e+05        548
1085    WEAK_FEATURE_2  2.684394e+05        938
940    ext_sources_sum  2.275616e+05        518
591       EXT_SOURCE_2  1.037206e+05        620


In [13]:
importance_df.to_excel('temp/feature_importance_big.xlsx', index=False)